In [1]:
import pandas as pd
import re
from urllib.parse import urlparse
import os
import numpy as np
import warnings
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv("C:/Users/USER/Desktop/URL-Phishing-Detection/datasets/raw_data/StealthPhisher2025.csv")

In [3]:
df.shape

(336749, 65)

In [4]:
df.columns

Index(['URL', 'LengthOfURL', 'Domain', 'URLComplexity', 'CharacterComplexity',
       'DomainLengthOfURL', 'IsDomainIP', 'TLD', 'TLDLength', 'LetterCntInURL',
       'URLLetterRatio', 'DigitCntInURL', 'URLDigitRatio', 'EqualCharCntInURL',
       'QuesMarkCntInURL', 'AmpCharCntInURL', 'OtherSpclCharCntInURL',
       'URLOtherSpclCharRatio', 'NumberOfHashtags', 'NumberOfSubdomains',
       'HavingPath', 'PathLength', 'HavingQuery', 'HavingFragment',
       'HavingAnchor', 'HasSSL', 'IsUnreachable', 'LineOfCode',
       'LongestLineLength', 'HasTitle', 'HasFavicon', 'HasRobotsBlocked',
       'IsResponsive', 'IsURLRedirects', 'IsSelfRedirects', 'HasDescription',
       'HasPopup', 'HasIFrame', 'IsFormSubmitExternal', 'HasSocialMediaPage',
       'HasSubmitButton', 'HasHiddenFields', 'HasPasswordFields',
       'HasBankingKey', 'HasPaymentKey', 'HasCryptoKey',
       'HasCopyrightInfoKey ', 'CntImages', 'CntFilesCSS', 'CntFilesJS',
       'CntSelfHRef', 'CntEmptyRef', 'CntExternalRef', 'Cn

In [5]:
df_extracted_feature = df[['URL', 'Label']]

In [6]:
df_extracted_feature.shape

(336749, 2)

In [7]:
df_extracted_feature.columns

Index(['URL', 'Label'], dtype='object')

---- Start Feature Extracting-------------

In [8]:
# Average_Word
words = df_extracted_feature["URL"].str.findall(r"[A-Za-z]+")
df_extracted_feature["Average_Word"] = words.apply(lambda x: sum(len(w) for w in x)/len(x) if len(x) > 0 else 0)

In [9]:
#Base64_Pattern_Cnt
df_extracted_feature['Base64_Pattern_Cnt'] = df['Base64PatternCnt']

In [10]:
#Character _Repetition
df_extracted_feature.loc[:, "Character_Repetition"] = df_extracted_feature["URL"].apply(
    lambda x: sum(len(m.group(0)) - 1 for m in re.finditer(r"(.)\1+", str(x)))
)

In [11]:
#Check EXE
df_extracted_feature["Check_EXE"] = df_extracted_feature["URL"].str.contains(r"\.exe\b", case=False, regex=True).astype(int)

In [12]:
#Count_-
df_extracted_feature["Count_-"] = df_extracted_feature["URL"].str.count("-")

In [13]:
#Count_%
df_extracted_feature["Count_%"] = df_extracted_feature["URL"].str.count("%")

In [14]:
#Count_&
df_extracted_feature["Count_&"] = df_extracted_feature["URL"].str.count("&")

In [15]:
#Count_/
df_extracted_feature["Count_/"] = df_extracted_feature["URL"].str.count("/")

In [16]:
#Count_;
df_extracted_feature["Count_;"] = df_extracted_feature["URL"].str.count(";")

In [17]:
#Count_?
df_extracted_feature["Count_?"] = df_extracted_feature["URL"].str.count(r"\?")

In [18]:
#Count_@
df_extracted_feature["Count_@"] = df_extracted_feature["URL"].str.count("@")

In [19]:
#Count_=
df_extracted_feature["Count_="] = df_extracted_feature["URL"].str.count("=")

In [20]:
#Count_Digit
df_extracted_feature["Count_Digit"] = df_extracted_feature["URL"].str.count(r"\d")

In [21]:
#Count_Dir  URL deki tüm sayıların toplamı
df_extracted_feature["Digits_Sum"] = df_extracted_feature["URL"].str.findall(r"\d").apply(lambda x: sum(map(int, x)) if x else 0)

In [22]:
#Count_Dot
df_extracted_feature["Count_Dot"] = df_extracted_feature["URL"].str.count(r"\.")

In [23]:
#Count_Embed_Domain
df_extracted_feature["Count_Embed_Domain"] = df_extracted_feature["URL"].apply(
    lambda x: str(x).lower().count("://") + str(x).lower().count("//") - 1
)

In [24]:
#Count_Letter
df_extracted_feature["Count_Letter"] = df_extracted_feature["URL"].str.count(r"[A-Za-z]")

In [25]:
#Count_Path_Slash
df_extracted_feature["Count_Path_Slash"] = df_extracted_feature["URL"].apply(lambda x: urlparse(str(x)).path.count("/"))

In [26]:
#Count_www
df_extracted_feature["Count_www"] = df_extracted_feature["URL"].str.count(r"www")

In [27]:
#Count_Http
df_extracted_feature["Count_Http"] = df_extracted_feature["URL"].str.count(r"http")

In [28]:
#Count_Https
df_extracted_feature["Count_Http"] = df_extracted_feature["URL"].str.count(r"htts")

In [29]:
#Digit/Letter
digits = df_extracted_feature["URL"].str.count(r"\d")
letters = df_extracted_feature["URL"].str.count(r"[A-Za-z]")
df_extracted_feature["Digit/Letter"] = digits / letters.replace(0, 1)

In [30]:
#Digit_Alphabet_Ratio
digits = df_extracted_feature["URL"].str.count(r"\d")
letters = df_extracted_feature["URL"].str.count(r"[A-Za-z]")
df_extracted_feature["Digit_Alphabet_Ratio"] = digits / letters.replace(0, 1)

In [31]:
#Domain_Length_Of_URL
df_extracted_feature["Domain_Length_Of_URL"] = df_extracted_feature["URL"].apply(
    lambda x: len(urlparse(x if (str(x).startswith('http://') or str(x).startswith('https://')) else 'http://' + str(x)).netloc)
)

In [32]:
#Domain_URL_Ratio
df_extracted_feature["Domain_URL_Ratio"] = df_extracted_feature["URL"].apply(
    lambda x: len(urlparse(x if (str(x).startswith(('http://', 'https://'))) else 'http://' + str(x)).netloc)
) / df_extracted_feature["URL"].str.len().replace(0, 1)

In [33]:
#Double_Slash_Count
df_extracted_feature["Double_Slash_Count"] = df_extracted_feature["URL"].str.count(r"//")

In [34]:
#fd_length
df_extracted_feature["fd_length"] = df_extracted_feature["URL"].apply(
    lambda x: len(next((p for p in urlparse(x if "://" in str(x) else "http://" + str(x)).path.split("/") if p), ""))
)

In [35]:
#File_Extension
df_extracted_feature["File_Extension"] = df_extracted_feature["URL"].apply(
    lambda x: os.path.splitext(urlparse(str(x)).path)[1])

In [36]:
#FractalDimension
df_extracted_feature['FractalDimension'] = df['FractalDimension']

In [37]:
#FTP_Used
df_extracted_feature["FTP_Used"] = df_extracted_feature["URL"].str.lower().str.startswith("ftp").astype(int)

In [38]:
#Having_Path
df_extracted_feature["Having_Path"] = df_extracted_feature["URL"].apply(
    lambda x: 1 if urlparse(x if "://" in str(x) else "http://" + str(x)).path not in ["", "/"] else 0
)

In [39]:
#Hex_Pattern_Cnt
df_extracted_feature["Hex_Pattern_Cnt"] = df_extracted_feature["URL"].str.count(r"%[0-9A-Fa-f]{2}")

In [40]:
#Host_Length
df_extracted_feature["Host_Length"] = df_extracted_feature["URL"].apply(
    lambda x: len(urlparse(x if "://" in str(x) else "http://" + str(x)).netloc)
)

In [41]:
#Host_Precense_Of_Digit
df_extracted_feature["Host_Precense_Of_Digit"] = df_extracted_feature["URL"].apply(
    lambda x: int(any(c.isdigit() for c in urlparse(x if "://" in str(x) else "http://" + str(x)).netloc))
)

In [42]:
#Kolmogorov_Complexity 
df_extracted_feature['Kolmogorov_Complexity'] = df['KolmogorovComplexity']

In [43]:
# Length_Of_URL
df_extracted_feature['Length_Of_URL'] =df['LengthOfURL'] 

In [44]:
#Longest_Word
df_extracted_feature["Longest_Word"] = df_extracted_feature["URL"].str.findall(r"[A-Za-z]+").apply(lambda x: max(map(len, x)) if x else 0)

In [45]:
#Longest_Word_in_Hostname
df_extracted_feature["Longest_Word_in_Hostname"] = df_extracted_feature["URL"].apply(
    lambda x: max([len(w) for w in re.findall(r"[A-Za-z]+", urlparse(x if str(x).startswith(('http://', 'https://')) else 'http://' + str(x)).netloc)] + [0])
)

In [46]:
# Path_Length
df_extracted_feature["Path_Length"] = df_extracted_feature["URL"].apply(
    lambda x: len(urlparse(x if "://" in str(x) else "http://" + str(x)).path)
)

In [47]:
#Port_Number
df_extracted_feature["Port_Number"] = df_extracted_feature["URL"].apply(
    lambda x: (p if (p := urlparse(x if "://" in str(x) else "http://" + str(x)).port) is not None else 0)
)

In [48]:
#Having_Query
df_extracted_feature["Having_Query"] = df_extracted_feature["URL"].apply(
    lambda x: int(bool(urlparse(x if "://" in str(x) else "http://" + str(x)).query))
)

In [49]:
#Query_Length
df_extracted_feature["Query_Length"] = df_extracted_feature["URL"].apply(
    lambda x: len(urlparse(x if "://" in str(x) else "http://" + str(x)).query)
)

In [50]:
#ShannonEntropy
df_extracted_feature["ShannonEntropy"] = df['ShannonEntropy']

In [51]:
#Short_Url
shorteners = r"(bit\.ly|goo\.gl|shorte\.st|go2l\.ink|x\.co|ow\.ly|t\.co|tinyurl\.com|tr\.im|is\.gd|cli\.gs|yfrog\.com|migre\.me|ff\.im|tiny\.cc|url4\.eu|twit\.ac|su\.pr|twurl\.nl|snipurl\.com|short\.to|budurl\.com|ping\.fm|post\.ly|just\.as|bkite\.com|snipr\.com|fic\.kr|loopt\.us|doiop\.com|short\.ie|kl\.am|wp\.me|rubyurl\.com|om\.ly|to\.ly|bit\.do|t\.ly|lnkd\.in|db\.tt|qr\.ae|adf\.ly|bitly\.com|cur\.lv|ity\.im|q\.gs|po\.st|bc\.vc)"
df_extracted_feature["Short_Url"] = df_extracted_feature["URL"].str.contains(shorteners, regex=True, case=False).astype(int)

In [52]:
#Special_Char_Alphabet_Ratio
letters = df_extracted_feature["URL"].str.count(r"[A-Za-z]")
special = df_extracted_feature["URL"].str.count(r"[^A-Za-z0-9]")

df_extracted_feature["Special_Char_Alphabet_Ratio"] = special / letters.replace(0, 1)

In [53]:
#STD_of_Words_Length
words = df_extracted_feature["URL"].str.findall(r"[A-Za-z]+")
df_extracted_feature["STD_of_Words_Length"] = words.apply(lambda x: np.std([len(w) for w in x]) if x else 0)

In [54]:
#Subdomain
df_extracted_feature["Subdomain"] = df_extracted_feature["URL"].apply(
    lambda x: max(len(urlparse(x if "://" in str(x) else "http://" + str(x)).netloc.split(".")) - 2, 0)
)

In [55]:
#Suffix
df_extracted_feature["Suffix"] = df_extracted_feature["URL"].apply(
    lambda x: urlparse(x if "://" in str(x) else "http://" + str(x)).netloc.split(".")[-1] 
    if "." in urlparse(x if "://" in str(x) else "http://" + str(x)).netloc else ""
)


In [56]:
#Tld_Length
df_extracted_feature['Tld_Length'] = df['TLDLength']

In [57]:
#Uppercase_Lowercase_Ratio
upper = df_extracted_feature["URL"].str.count(r"[A-Z]")
lower = df_extracted_feature["URL"].str.count(r"[a-z]")
df_extracted_feature["Uppercase_Lowercase_Ratio"] = upper / lower.replace(0, 1)

In [58]:
#URL/Path
path_len = df_extracted_feature["URL"].apply(lambda x: len(urlparse(x if "://" in str(x) else "http://" + str(x)).path))
url_len = df_extracted_feature["URL"].str.len()

df_extracted_feature["URL/Path"] = url_len / path_len.replace(0, 1)

In [59]:
#use_of_ip_address
df_extracted_feature["use_of_ip_address"] = df_extracted_feature["URL"].apply(
    lambda x: int(bool(re.search(r"(\d{1,3}\.){3}\d{1,3}", urlparse(x if "://" in str(x) else "http://" + str(x)).netloc)))
)

In [60]:
#Vowel/Consonant
vowels = df_extracted_feature["URL"].str.count(r"[aeiouAEIOU]")
letters = df_extracted_feature["URL"].str.count(r"[A-Za-z]")
consonants = letters - vowels

df_extracted_feature["Vowel/Consonant"] = vowels / consonants.replace(0, 1)

In [61]:
df_extracted_feature.columns

Index(['URL', 'Label', 'Average_Word', 'Base64_Pattern_Cnt',
       'Character_Repetition', 'Check_EXE', 'Count_-', 'Count_%', 'Count_&',
       'Count_/', 'Count_;', 'Count_?', 'Count_@', 'Count_=', 'Count_Digit',
       'Digits_Sum', 'Count_Dot', 'Count_Embed_Domain', 'Count_Letter',
       'Count_Path_Slash', 'Count_www', 'Count_Http', 'Digit/Letter',
       'Digit_Alphabet_Ratio', 'Domain_Length_Of_URL', 'Domain_URL_Ratio',
       'Double_Slash_Count', 'fd_length', 'File_Extension', 'FractalDimension',
       'FTP_Used', 'Having_Path', 'Hex_Pattern_Cnt', 'Host_Length',
       'Host_Precense_Of_Digit', 'Kolmogorov_Complexity', 'Length_Of_URL',
       'Longest_Word', 'Longest_Word_in_Hostname', 'Path_Length',
       'Port_Number', 'Having_Query', 'Query_Length', 'ShannonEntropy',
       'Short_Url', 'Special_Char_Alphabet_Ratio', 'STD_of_Words_Length',
       'Subdomain', 'Suffix', 'Tld_Length', 'Uppercase_Lowercase_Ratio',
       'URL/Path', 'use_of_ip_address', 'Vowel/Consonant'],


In [62]:
df_extracted_feature.shape

(336749, 54)

In [63]:
df_extracted_feature

,URL,Label,Average_Word,Base64_Pattern_Cnt,Character_Repetition,Check_EXE,Count_-,Count_%,Count_&,Count_/,...,Short_Url,Special_Char_Alphabet_Ratio,STD_of_Words_Length,Subdomain,Suffix,Tld_Length,Uppercase_Lowercase_Ratio,URL/Path,use_of_ip_address,Vowel/Consonant
0,https://bafkreibre4pwizu3d73y7at37ewy6nhklfhb4...,Phishing,3.529412,3,3,0,1,0,0,2,...,0,0.116667,2.379265,2,com,3,0.000,84.000000,0,0.333333
1,http://101.200.220.118:8090/ledshow2.exe,Phishing,4.666667,3,5,1,0,0,0,3,...,0,0.642857,1.699673,2,118:8090,8,0.000,3.076923,1,0.400000
2,https://1llc5nv.duckdns.org/,Phishing,4.000000,2,3,0,0,0,0,3,...,0,0.300000,1.788854,1,org,3,0.000,28.000000,0,0.111111
3,http://hrga.melonwoodhomes.com/,Phishing,6.250000,2,3,0,0,0,0,3,...,0,0.240000,4.493050,1,com,3,0.000,31.000000,0,0.470588
4,https://www.aspb.gob.bo,Legitimate,3.400000,2,4,0,0,0,0,2,...,0,0.352941,1.019804,2,bo,2,0.000,23.000000,0,0.214286
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
336744,https://www.theboothcompany.ca,Legitimate,6.250000,2,5,0,0,0,0,2,...,0,0.200000,5.165995,1,ca,2,0.000,30.000000,0,0.315789
336745,https://www.biscaynetimes.com,Legitimate,6.000000,2,4,0,0,0,0,2,...,0,0.208333,4.123106,1,com,3,0.000,29.000000,0,0.333333
336746,https://sbvkt5.firebaseapp.com/,Phishing,6.000000,2,3,0,0,0,0,3,...,0,0.250000,3.000000,1,com,3,0.000,31.000000,0,0.333333
336747,http://117.219.119.58:47440/Mozi.a,Phishing,3.000000,3,5,0,0,0,0,3,...,0,1.000000,1.414214,2,58:47440,8,0.125,4.857143,1,0.500000


In [66]:
df_extracted_feature.to_csv("datasets/features_dataset.csv", index=False)

In [67]:
df_extracted_feature.to_excel("datasets/features_dataset.xlsx", index=False)
